# Fietsstallingen pipelines

Load and validate the revised definition once, then compile its publisher and consumer as separate pipeline builds. Each build is written to its own output directory so their generated files do not overwrite each other.

In [1]:
from pathlib import Path

from rdflib import Graph
from rdfine import GraphReader
from compilers import PipelineGenerator, ProjectBuilder

In [2]:
data_dir = Path("../data")
source_files = [
    "catalog/catalog-core.ttl",
    "catalog/catalog-ldio.ttl",
    "catalog/catalog-nifi.ttl",
    "catalog/catalog-rdfc.ttl",
    "catalog/catalog-rdfc-manual.ttl",
    "catalog/catalog-sw.ttl",
    "pipelines/pipeline_definition_fietsstallingen.revised.ttl",
    "catalog/catalog-application-profile-shapes.ttl",
]

graph = Graph()
for filename in source_files:
    graph.parse(data_dir / filename, publicID="file:///workspace/pipeline/")

reader = GraphReader(graph)
for rules in ("inference_rules/inference_rules.yaml", "inference_rules/rdfc_inference_rules.yaml"):
    reader = reader.infer(data_dir / rules)

In [3]:
report = reader.validate(advanced=True, inference="rdfs")
violations = report.select(
    "?focus ?message",
    """
    ?result a sh:ValidationResult ;
        sh:focusNode ?focus ;
        sh:resultMessage ?message .
    """,
)

if not report.ask("?report sh:conforms true"):
    for row in violations.itertuples(index=False):
        print(f"{row.focus}: {row.message}")
    raise ValueError("Fietsstallingen definitions do not conform")

print("The revised fietsstallingen definitions conform.")

The revised fietsstallingen definitions conform.


`PipelineGenerator` compiles one `tcs:PipelineDefinition` at a time. Run it once for the publisher and once for the consumer, retaining both results for inspection.

In [4]:
pipeline_ids = {
    "publisher": "fs:PublisherPipeline",
    "consumer": "fs:ConsumerPipeline",
}

generators = {}
build_graphs = {}
builders = {}

for name, pipeline_id in pipeline_ids.items():
    generator = PipelineGenerator(pipeline_id, reader.graph)
    build_graph = generator.compile()

    generators[name] = generator
    build_graphs[name] = build_graph
    builders[name] = ProjectBuilder(build_graph)

    compiler_names = [compiler.__name__ for compiler in generator.compilers]
    print(f"{name}: {', '.join(compiler_names)}")

publisher: PipelineSeeder, PipelineEnricher, PipelineAssembler, GraphReducer, SegmentTagger, SemanticWorksEnvVarCompiler, LdioConfigCompiler, ValidationReportCompiler, DockerComposeCompiler


consumer: PipelineSeeder, PipelineEnricher, PipelineAssembler, BridgeTransportCompiler, GraphReducer, SegmentTagger, RdfcDockerFileCompiler, SemanticWorksEnvVarCompiler, VirtuosoCompiler, LdioConfigCompiler, RdfcConfigCompiler, ValidationReportCompiler, DockerComposeCompiler


In [5]:
for name, builder in builders.items():
    print(f"\n{name}:")
    for row in builder.files.itertuples(index=False):
        print(f"  {row.filepath}/{row.filename}")


publisher:
  ldio/application.yml
  ./docker-compose.yml
  validation/validation-report.ttl
  ldio/pipelines/fietsstallingen-pipeline.yml

consumer:
  ldio/application.yml
  ./docker-compose.yml
  rdfc/Dockerfile
  validation/validation-report.ttl
  rdfc/package.json
  semantic-works/config/virtuoso/virtuoso.ini
  ldio/pipelines/ldes-fietsstallingen-to-virtuoso-pipeline.yml
  rdfc/pyproject.toml
  rdfc/pipeline.ttl


Materialize the two builds below `out/fietsstallingen/`. Existing generated files at the same paths are overwritten by `ProjectBuilder`.

In [6]:
output_root = Path("../out/fietsstallingen")
written = {
    name: builder.write(output_root / name)
    for name, builder in builders.items()
}

for name, paths in written.items():
    print(f"\n{name}:")
    for path in paths:
        print(f"  {path}")


publisher:
  C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\fietsstallingen\publisher\ldio\application.yml
  C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\fietsstallingen\publisher\docker-compose.yml
  C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\fietsstallingen\publisher\validation\validation-report.ttl
  C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\fietsstallingen\publisher\ldio\pipelines\fietsstallingen-pipeline.yml

consumer:
  C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\fietsstallingen\consumer\ldio\application.yml
  C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\fietsstallingen\consumer\docker-compose.yml
  C:\Users\ThomasDelaeter\Documents\Projecten\toolchain-specification\pipeline generator\out\fietsstallingen